In [18]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.graphics.shapes import Drawing, Circle
from reportlab.graphics.charts.barcharts import VerticalBarChart
from reportlab.graphics.charts.piecharts import Pie
from reportlab.graphics.charts.textlabels import Label
from reportlab.graphics.barcode import code39
from datetime import datetime

def desenhar_fatura(pdf, cliente, produtos, numero_fatura):
    # Título da fatura com número
    pdf.setFont("Helvetica-Bold", 20)
    pdf.drawString(200, 750, f"Fatura {numero_fatura}")

    # Linha abaixo do título "Fatura"
    pdf.line(50, 745, 580, 745)

    # Data e hora da criação
    data_hora = datetime.now().strftime("%d/%m/%Y %H:%M")
    pdf.setFont("Helvetica", 10)
    pdf.drawString(450, 750, f"Data: {data_hora}")

    # Código do cliente (CPF/CNPJ)
    pdf.setFont("Helvetica", 12)
    pdf.drawString(50, 735, f"Código do Cliente: {cliente['codigo']}")

    # Dados do cliente
    pdf.setFont("Helvetica", 12)
    pdf.drawString(50, 720, f"Cliente: {cliente['nome']}")
    pdf.drawString(50, 705, f"Endereço: {cliente['endereco']}")
    pdf.drawString(50, 690, f"Telefone: {cliente['telefone']}")
    pdf.drawString(50, 675, f"Email: {cliente['email']}")

    # Cabeçalhos da tabela de produtos
    pdf.setFont("Helvetica-Bold", 12)
    pdf.drawString(50, 640, "Produto")
    pdf.drawString(250, 640, "Quantidade")
    pdf.drawString(350, 640, "Unidade")
    pdf.drawString(450, 640, "Preço Unitário")
    pdf.drawString(550, 640, "Subtotal")

    # Linha para separar cabeçalhos dos dados
    pdf.line(50, 635, 580, 635)

    # Adicionando produtos à tabela com linhas de separação e alinhamento
    y = 620
    total = 0
    quantidades = []
    nomes_produtos = []
    for produto in produtos:
        pdf.setFont("Helvetica", 12)
        pdf.drawString(50, y, produto['nome'])
        pdf.drawRightString(290, y, str(produto['quantidade']))
        pdf.drawString(350, y, produto['unidade'])
        pdf.drawRightString(490, y, f"R$ {produto['preco_unitario']:.2f}")
        subtotal = produto['quantidade'] * produto['preco_unitario']
        pdf.drawRightString(590, y, f"R$ {subtotal:.2f}")
        total += subtotal

        # Coletar dados para o gráfico
        quantidades.append(produto['quantidade'])
        nomes_produtos.append(produto['nome'])

        # Linhas horizontais para separar os produtos
        pdf.line(50, y - 5, 580, y - 5)
        y -= 20

    # Linha para separar subtotal e total
    pdf.line(50, y - 10, 580, y - 10)

    # Exibir total geral
    pdf.setFont("Helvetica-Bold", 14)
    pdf.drawString(400, y - 30, "Total Geral : ")
    pdf.drawRightString(590, y - 30, f"R$ {total:.2f}")

    # Gerar código de barras
    codigo_barra_valor = f"{cliente['codigo']}{numero_fatura}"
    barcode = code39.Standard39(codigo_barra_valor, barHeight=20, stop=1)
    barcode.drawOn(pdf, 50, y - 60)

    # Adicionar gráficos
    desenhar_grafico_barras(pdf, quantidades, nomes_produtos, y - 250)
    desenhar_grafico_donuts(pdf, quantidades, nomes_produtos, y - 500)

    # Finaliza o PDF
    pdf.save()

def desenhar_grafico_barras(pdf, quantidades, nomes_produtos, y_pos):
    drawing = Drawing(400, 200)
    bc = VerticalBarChart()
    bc.x = 50
    bc.y = 50
    bc.height = 125
    bc.width = 300
    bc.data = [quantidades]
    bc.strokeColor = colors.black

    bc.categoryAxis.labels.boxAnchor = 'n'
    bc.categoryAxis.labels.dx = 0
    bc.categoryAxis.labels.dy = -2
    bc.categoryAxis.labels.angle = 0
    bc.categoryAxis.categoryNames = nomes_produtos

    bc.valueAxis.valueMin = 0
    bc.valueAxis.valueMax = max(quantidades) * 1.2
    bc.valueAxis.valueStep = max(1, max(quantidades) // 5)

    # Título do gráfico
    titulo = Label()
    titulo.setOrigin(200, 180)
    titulo.setText("Quantidade de Produtos")
    titulo.fontSize = 14
    titulo.fontName = 'Helvetica-Bold'
    drawing.add(titulo)

    drawing.add(bc)
    drawing.drawOn(pdf, 50, y_pos)

def desenhar_grafico_donuts(pdf, quantidades, nomes_produtos, y_pos):
    drawing = Drawing(400, 200)
    pc = Pie()
    pc.x = 150
    pc.y = 50
    pc.width = 150
    pc.height = 150
    pc.data = quantidades
    pc.labels = nomes_produtos
    pc.startAngle = 90
    pc.slices.strokeWidth = 0.5

    # Desenhar o gráfico de pizza
    drawing.add(pc)

    # Simular o efeito de donut desenhando um círculo branco no centro
    hole = Circle(pc.x + pc.width / 2, pc.y + pc.height / 2, pc.width / 4)
    hole.fillColor = colors.white
    drawing.add(hole)

    # Título do gráfico
    titulo = Label()
    titulo.setOrigin(100, 90)
    titulo.setText("Distribuição de Quantidades")
    titulo.fontSize = 14
    titulo.fontName = 'Helvetica-Bold'
    drawing.add(titulo)

    drawing.drawOn(pdf, 50, y_pos)

# Dados fictícios para o exemplo
cliente = {
    "codigo": "123.456.789-00",  # CPF/CNPJ fictício
    "nome": "João da Silva",
    "endereco": "Rua Exemplo, 123, São Paulo, SP",
    "telefone": "(11) 98765-4321",
    "email": "joao.silva@email.com"
}

produtos = [
    {"nome": "Produto A", "quantidade": 2, "unidade": "kg", "preco_unitario": 10.00},
    {"nome": "Produto B", "quantidade": 3, "unidade": "unidade", "preco_unitario": 5.50},
    {"nome": "Produto C", "quantidade": 1.5, "unidade": "litro", "preco_unitario": 7.30},
    {"nome": "Produto D", "quantidade": 10, "unidade": "caixa", "preco_unitario": 12.40},
    {"nome": "Produto E", "quantidade": 4, "unidade": "pacote", "preco_unitario": 8.25},
]

numero_fatura = "2024-001"

# Criar o PDF
pdf = canvas.Canvas("fatura_exemplo_profissional_com_graficos.pdf", pagesize=letter)

desenhar_fatura(pdf, cliente, produtos, numero_fatura)
